# Set environment

In [1]:
### Basic
import numpy  as np
import scipy  as sc
import pandas as pd
import glob, os, re
import pickle, sys

### directory
DAT_DIR = "/data/clintko/flow_EQAPOL_normal/mpsk"

In [2]:
%load_ext rpy2.ipython
%matplotlib inline

In [3]:
%%R

library("tidyverse")
library("MPSK")
DAT_DIR = "/data/clintko/flow_EQAPOL_normal/mpsk"

/local_data/env-py3/lib/python3.6/site-packages/rpy2/rinterface/__init__.py:145: RRuntimeWarning: ── Attaching packages ─────────────────────────────────────── tidyverse 1.2.1 ──

  warnings.warn(x, RRuntimeWarning)
/local_data/env-py3/lib/python3.6/site-packages/rpy2/rinterface/__init__.py:145: RRuntimeWarning: ✔ ggplot2 3.1.0     ✔ purrr   0.2.5
✔ tibble  1.4.2     ✔ dplyr   0.7.8
✔ tidyr   0.8.2     ✔ stringr 1.3.1
✔ readr   1.1.1     ✔ forcats 0.3.0

  warnings.warn(x, RRuntimeWarning)
/local_data/env-py3/lib/python3.6/site-packages/rpy2/rinterface/__init__.py:145: RRuntimeWarning: ── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()

  warnings.warn(x, RRuntimeWarning)


# Load MPSK model

In [4]:
!ls /data/clintko/flow_EQAPOL_normal/mpsk

C_1e4.csv		 mpsk_1e4_costim_chain.RDS  Y_1e4_cmv.csv
mpsk_1e4_cmv_cal.RDS	 mpsk_1e4_costim_res.RDS    Y_1e4_costim.csv
mpsk_1e4_cmv_chain.RDS	 mpsk_1e4_seb_cal.RDS	    Y_1e4_seb.csv
mpsk_1e4_cmv_res.RDS	 mpsk_1e4_seb_chain.RDS
mpsk_1e4_costim_cal.RDS  mpsk_1e4_seb_res.RDS


read label

In [5]:
%%R

### read file
fname = "C_1e4.csv"
fpath = file.path(DAT_DIR, fname)
C     = readr::read_csv(fpath, col_names = FALSE)

### arrange the label into vector
C = C$X1
print(dim(C))    # NULL
print(length(C)) # 110,000

/local_data/env-py3/lib/python3.6/site-packages/rpy2/rinterface/__init__.py:145: RRuntimeWarning: Parsed with column specification:
cols(
  X1 = col_integer()
)

  warnings.warn(x, RRuntimeWarning)


NULL
[1] 110000


In [6]:
%%R

chainSummary_costim  = readRDS(file.path(DAT_DIR, "mpsk_1e4_costim_chain.RDS"))
chainSummary_cmv     = readRDS(file.path(DAT_DIR, "mpsk_1e4_cmv_chain.RDS"))
chainSummary_seb     = readRDS(file.path(DAT_DIR, "mpsk_1e4_seb_chain.RDS"))

In [7]:
%%R

NSIZE = 10000
SEED  = 123
J = length(unique(C))

# Simulate data

In [21]:
%%R

generate_sample = function(chainSummary, J){
    kIdx = unique(chainSummary$t)
    genY = NULL # data
    genC = NULL # label --- sample / lab
    genT = NULL # label --- cluster w/in a sample

    ### k --- cluster ; j --- sample / lab
    for (k in 1:length(kIdx)) { # for each cluster
    
        ### acquire the corresponding cluster idx from the fitted data
        kk = kIdx[k]
    
        for (j in 1:J) { # for each sample 
            ### proportion / weight of a cluster
            #njk = floor(sum(C == j) * chainSummary$W[j, kk])
            njk = ceiling(sum(C == j) * chainSummary$W[j, kk])
        
            ### generate a cluster from skew normal
            genY = rbind(genY, 
                      ### skew normal distribution
                      sn::rmsn(
                          n     = njk, 
                          xi    = chainSummary$xi[j,,kk],
                          Omega = chainSummary$Omega[,,kk],
                          alpha = chainSummary$alpha[,kk]))
        
            ### label of sample / lab
            genC = c(genC, rep(j, njk))
        
            ### label of cluster
            genT = c(genT, rep(kk, njk))
        } # end inner for
    } # end outer for
    
    ### sample rows to make final data (NSIZE * 11, 14)
    genY2 = lapply(unique(C), function(idx_sample){
        idx = (genC == idx_sample)
        Y = genY[idx,]
        idx = sample(nrow(Y), size = NSIZE, replace = TRUE)
        return(Y[idx,])
    }) # sapply
    
    return(do.call(rbind, genY2))
} # end func

generate costim

In [62]:
dat = []
%R set.seed(123)

for _ in range(10):
    Y = %R generate_sample(chainSummary_costim, J)
    Y = Y.reshape(11, 10000, 14)
    dat.append(Y)

dat_costim = np.concatenate(dat)

In [63]:
dat_costim.shape

(110, 10000, 14)

recheck the seed and reshape

In [64]:
%R set.seed(123)
Y = %R generate_sample(chainSummary_costim, J)
Y = Y.reshape(11, 10000, 14)

In [65]:
np.allclose(dat_costim[0], Y[0])

True

generate CMV

In [67]:
dat = []
%R set.seed(123)

for _ in range(10):
    Y = %R generate_sample(chainSummary_cmv, J)
    Y = Y.reshape(11, 10000, 14)
    dat.append(Y)

dat_cmv = np.concatenate(dat)

generate SEB

In [68]:
dat = []
%R set.seed(123)

for _ in range(10):
    Y = %R generate_sample(chainSummary_seb, J)
    Y = Y.reshape(11, 10000, 14)
    dat.append(Y)

dat_seb = np.concatenate(dat)

# Combine

In [69]:
label = [0] * dat_costim.shape[0] + \
        [1] * dat_cmv.shape[0]    + \
        [2] * dat_seb.shape[0]
label = np.array(label)
print(label.shape)

(330,)


In [70]:
matrix = np.concatenate([dat_costim, dat_cmv, dat_seb])
print(matrix.shape)

(330, 10000, 14)


wrap in dictionary

In [71]:
data = dict()
data["label"]  = label
data["matrix"] = matrix

store the results

In [73]:
### set up a pickle file
fname = "mpsk_1e4_simulation.P"
fpath = os.path.join(DAT_DIR, fname)
file = open(fpath, 'wb')

### store the dictionary
pickle.dump(data, file)
file.close()

In [74]:
!ls /data/clintko/flow_EQAPOL_normal/mpsk

C_1e4.csv		 mpsk_1e4_costim_chain.RDS  mpsk_1e4_simulation.P
mpsk_1e4_cmv_cal.RDS	 mpsk_1e4_costim_res.RDS    Y_1e4_cmv.csv
mpsk_1e4_cmv_chain.RDS	 mpsk_1e4_seb_cal.RDS	    Y_1e4_costim.csv
mpsk_1e4_cmv_res.RDS	 mpsk_1e4_seb_chain.RDS     Y_1e4_seb.csv
mpsk_1e4_costim_cal.RDS  mpsk_1e4_seb_res.RDS
